In [1]:
grid = 256

In [2]:
shanghai_city = r'D:\shanghai workshop\vector\Urban Center_final ver\Urban Center_final ver.shp'

# 网格

In [6]:
import geopandas as gpd
from shapely.geometry import box
import numpy as np

def export_bounding_rectangle(input_shapefile, output_shapefile, grid_size=256):
    """
    导出对齐到 UTM 网格的最小外接矩形边界
    """
    # 读取原始数据
    gdf = gpd.read_file(input_shapefile)

    # 转换为 UTM 坐标系（上海：EPSG:32651）
    gdf = gdf.to_crs(epsg=32651)

    # 获取原始边界
    minx, miny, maxx, maxy = gdf.total_bounds

    # ✅ 对齐到指定网格尺寸（例如 256m）
    minx_aligned = np.floor(minx / grid_size) * grid_size
    miny_aligned = np.floor(miny / grid_size) * grid_size
    maxx_aligned = np.ceil(maxx / grid_size) * grid_size
    maxy_aligned = np.ceil(maxy / grid_size) * grid_size

    # 构造矩形
    rectangle = box(minx_aligned, miny_aligned, maxx_aligned, maxy_aligned)

    # 保存为新的 shapefile
    out_gdf = gpd.GeoDataFrame(geometry=[rectangle], crs=gdf.crs)
    out_gdf.to_file(output_shapefile)

    print(f"✅ 已保存对齐的矩形边界：{output_shapefile}")
    print(f"📐 对齐范围：{minx_aligned}, {miny_aligned}, {maxx_aligned}, {maxy_aligned}")

# 示例调用
export_bounding_rectangle(
    input_shapefile=shanghai_city,
    output_shapefile=r"D:\shanghai workshop\vector\shanghai_rectangle_boundary.shp",
    grid_size=256  # 单位：米
)


✅ 已保存对齐的矩形边界：D:\shanghai workshop\vector\shanghai_rectangle_boundary.shp
📐 对齐范围：314368.0, 3395328.0, 428288.0, 3483136.0


In [8]:
import geopandas as gpd
from shapely.geometry import box
import numpy as np

def generate_aligned_grid(input_shapefile, output_shapefile, grid_size):
    # 读取 polygon（例如上海边界）
    gdf = gpd.read_file(input_shapefile)

    # 转为 UTM 坐标系（上海一般是 EPSG:32651）
    gdf = gdf.to_crs(epsg=32651)

    # 获取边界
    bounds = gdf.total_bounds
    minx, miny, maxx, maxy = bounds

    # ✅ 对齐到 UTM 网格起点（强制从网格对齐点开始）
    minx_aligned = np.floor(minx / grid_size) * grid_size
    miny_aligned = np.floor(miny / grid_size) * grid_size
    maxx_aligned = np.ceil(maxx / grid_size) * grid_size
    maxy_aligned = np.ceil(maxy / grid_size) * grid_size

    # 生成网格
    grid_cells = []
    id_counter = 0
    for x0 in np.arange(minx_aligned, maxx_aligned, grid_size):
        for y0 in np.arange(miny_aligned, maxy_aligned, grid_size):
            x1 = x0 + grid_size
            y1 = y0 + grid_size
            cell = box(x0, y0, x1, y1)
            grid_cells.append({"geometry": cell, "ID": id_counter})
            id_counter += 1

    grid_gdf = gpd.GeoDataFrame(grid_cells, crs=gdf.crs)

    # ✅ 裁剪 polygon 内部
    grid_clipped = gpd.overlay(grid_gdf, gdf, how='intersection')

    # 保存
    grid_clipped.to_file(output_shapefile)
    print(f"✅ 生成完成：{output_shapefile}")
    print(f"📌 网格数量：{len(grid_clipped)}，坐标系：{grid_clipped.crs}")

# 示例调用
generate_aligned_grid(
    input_shapefile=r"D:\shanghai workshop\vector\shanghai_rectangle_boundary.shp",
    output_shapefile=r"D:\shanghai workshop\vector\shanghai_grid_256m.shp",
    grid_size=256
)


✅ 生成完成：D:\shanghai workshop\vector\shanghai_grid_256m.shp
📌 网格数量：152635，坐标系：EPSG:32651


# 上海市切分

In [9]:
import geopandas as gpd

shanghai_polygon = gpd.read_file(shanghai_city)
grid = gpd.read_file(r"D:\shanghai workshop\vector\shanghai_grid_256m.shp")

# 坐标系统一
if shanghai_polygon.crs != grid.crs:
    grid = grid.to_crs(shanghai_polygon.crs)

# 修复 geometry 问题
shanghai_polygon["geometry"] = shanghai_polygon.buffer(0)
grid = grid[grid.is_valid]

# 空间筛选
grid_intersect = grid[grid.geometry.intersects(shanghai_polygon.unary_union)]

# 输出
output_path = r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp"
grid_intersect.to_file(output_path)

print(f"✅ 已保存筛选后的网格文件，路径：{output_path}")
print(f"筛选后保留格子数：{len(grid_intersect)}")

C:\Users\owner\AppData\Local\Temp\ipykernel_5048\3905440531.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  grid_intersect = grid[grid.geometry.intersects(shanghai_polygon.unary_union)]


✅ 已保存筛选后的网格文件，路径：D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp
筛选后保留格子数：33857


# 01 park

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_park_by_grid(parks_shp, grid_shp, output_folder, resolution=10):
    # 读取数据
    parks = gpd.read_file(parks_shp)
    grid = gpd.read_file(grid_shp)

    # 坐标系统一（建议米制）
    if not parks.crs.is_projected:
        parks = parks.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(parks.crs)
    elif parks.crs != grid.crs:
        grid = grid.to_crs(parks.crs)

    # 清理空 geometry
    parks = parks[~parks.geometry.is_empty & parks.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_folder, exist_ok=True)

    for _, row in grid.iterrows():
        tile_id = row["ID"]  # 使用网格的 ID 字段作为文件名
        grid_geom = row.geometry

        # 提取该格网内的公园
        clipped = gpd.overlay(parks, gpd.GeoDataFrame(geometry=[grid_geom], crs=parks.crs), how="intersection")

        if clipped.empty:
            continue  # 如果该区域没有公园，跳过

        # 计算 raster 尺寸
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 准备 raster 数据
        shapes = [(geom, 1) for geom in clipped.geometry]
        raster = rasterize(
            shapes,
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype="uint8"
        )

        # 输出路径
        out_path = os.path.join(output_folder, f"park_tile_{tile_id}.tif")

        # 写入 raster
        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=parks.crs,
            transform=transform
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ Saved: {out_path}")

# 示例调用
rasterize_park_by_grid(
    parks_shp=r"D:\shanghai workshop\building+park+waterbody\landuse-公园用地\landuse_park.shp",
    grid_shp=rf"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    output_folder=r"D:\shanghai workshop\raster\final_park_tiles",
    resolution=2  # 2米分辨率
)

✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_196.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_532.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_539.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_882.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_883.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1225.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1226.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1568.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1569.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3631.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3641.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3974.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3975.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\pa

# 02 water

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_fclass_by_grid(shapefile_path, grid_path, fclass_mapping, output_dir, resolution):
    # 读取水体数据和网格
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统一为 UTM 51N
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 保留指定类别的记录，并转换为数值
    gdf = gdf[gdf['fclass'].isin(fclass_mapping.keys())]
    gdf['class_val'] = gdf['fclass'].map(fclass_mapping)

    # 清理空 geometry
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_dir, exist_ok=True)

    for _, cell in grid.iterrows():
        tile_id = cell["ID"]  # 使用 ID 字段作为输出文件名
        grid_geom = cell.geometry

        # 裁剪出当前格子内的水体
        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")

        if clipped.empty:
            continue

        # 计算输出 raster 尺寸
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 构造 shapes
        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=255,  # Nodata 为 255
            dtype='uint8'
        )

        out_path = os.path.join(output_dir, f"water_tile_{tile_id}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=255
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 示例调用
fclass_mapping = {
    'dock': 0,
    'reservoir': 1,
    'riverbank': 2,
    'water': 3,
    'wetland': 4
}

rasterize_fclass_by_grid(
    shapefile_path=r"D:\BaiduNetdiskDownload\3-shanghai-latest-free.shp\gis_osm_water_a_free_1.shp",
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    fclass_mapping=fclass_mapping,
    output_dir=r"D:\shanghai workshop\raster\final_water_tiles",
    resolution=2
)

# 03 building

In [5]:
import rasterio
import geopandas as gpd

raster = rasterio.open(r'D:\shanghai workshop\raster\building.tif')
print("栅格坐标系:", raster.crs)

grid = gpd.read_file(r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp")
print("网格坐标系:", grid.crs)


栅格坐标系: PROJCS["UTM_Modified",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",121.4348038],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
网格坐标系: EPSG:4326


In [1]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask
import os

def split_raster_by_grid(original_raster, grid_path, output_dir):
    # 读取矢量网格
    grid = gpd.read_file(grid_path)

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(original_raster) as src:
        raster_crs = src.crs
        print("📌 原始栅格坐标系:", raster_crs)

        # 坐标系统一
        if grid.crs != raster_crs:
            print("🔄 正在将矢量网格投影为与栅格一致的坐标系...")
            grid = grid.to_crs(raster_crs)

        for _, row in grid.iterrows():
            tile_id = row["ID"]  # 使用 grid 中的 ID 字段
            geom = [row.geometry]

            try:
                out_image, out_transform = mask(src, geom, crop=True)
            except Exception as e:
                print(f"⚠️ 跳过 ID={tile_id}，裁剪失败: {e}")
                continue

            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            out_path = os.path.join(output_dir, f"building_tile_{tile_id}.tif")
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)

            print(f"✅ 已保存切块: {out_path}")

# 示例调用
split_raster_by_grid(
    original_raster=r'D:\shanghai workshop\raster\building.tif',
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    output_dir=r"D:\shanghai workshop\raster\final_building_tiles"
)


📌 原始栅格坐标系: PROJCS["UTM_Modified",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",121.4348038],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
🔄 正在将矢量网格投影为与栅格一致的坐标系...
⚠️ 跳过 ID=47，裁剪失败: Input shapes do not overlap raster.
⚠️ 跳过 ID=48，裁剪失败: Input shapes do not overlap raster.
⚠️ 跳过 ID=49，裁剪失败: Input shapes do not overlap raster.
⚠️ 跳过 ID=180，裁剪失败: Input shapes do not overlap raster.
⚠️ 跳过 ID=181，裁剪失败: Input shapes do not overlap raster.
⚠️ 跳过 ID=182，裁剪失败: Input shapes do not overlap raster.
⚠️ 跳过 ID=183，裁剪失败: Input shapes do not overlap raster.
⚠️ 跳过 ID=184，裁剪失败: Input shapes do 

# 04 road

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_by_grid_with_auto_class_mapping(shapefile_path, grid_path, class_field, output_dir, resolution=10, nodata=255):
    # 读取数据
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统统一为投影坐标系（UTM 51N）
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 清理无效几何
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    # 检查字段是否存在
    if class_field not in gdf.columns:
        raise ValueError(f"字段 {class_field} 不存在于 shapefile 中")

    # 获取字段的唯一值列表，并生成映射字典（类别字符串->数字ID，从1开始，0 留作背景）
    unique_classes = sorted(gdf[class_field].dropna().unique())
    class_to_id = {cls: idx+1 for idx, cls in enumerate(unique_classes)}
    print(f"自动生成类别映射（{class_field}）：{class_to_id}")

    # 用映射替换原字段，未包含的赋值0
    gdf['class_val'] = gdf[class_field].map(class_to_id).fillna(0).astype(int)

    os.makedirs(output_dir, exist_ok=True)

    for idx, cell in grid.iterrows():
        grid_geom = cell.geometry

        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")
        if clipped.empty:
            continue

        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=nodata,
            dtype='uint8'
        )
        tile_id = cell["ID"]
        out_path = os.path.join(output_dir, f"tile_{tile_id}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=nodata
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 使用示例
rasterize_by_grid_with_auto_class_mapping(
    shapefile_path=r"D:\shanghai workshop\vector\Shanghai street_ver3\Shanghai street_ver3.shp",
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    class_field="Type",
    output_dir=r"D:\shanghai workshop\raster\final_street_tiles",
    resolution=1,
    nodata=255
)

# population

In [5]:
import geopandas as gpd

population_data = r'D:\shanghai workshop\building+park+waterbody\【25028】基于手机信令的上海市人口分布数据\[20250331]基于手机信令的上海市人口分布数据（2023年）\home.shp'
gpd.read_file(population_data)

,FID_1,LAT,COV,CITYCODE,WGS84_Lat,WGS84_Lng,LON,geometry
0,18229,31.2625,99085,83101,31.264411,121.462954,121.4675,POINT (121.4675 31.2625)
1,18230,31.3625,93601,83101,31.364291,121.377847,121.3825,POINT (121.3825 31.3625)
2,18231,31.3475,72602,83101,31.349646,121.573274,121.5775,POINT (121.5775 31.3475)
3,18232,31.2175,65892,83101,31.219432,121.462958,121.4675,POINT (121.4675 31.2175)
4,18233,31.2925,63597,83101,31.294470,121.488036,121.4925,POINT (121.4925 31.2925)
...,...,...,...,...,...,...,...,...
3565,35784,31.2375,229,83205,31.239373,121.052932,121.0575,POINT (121.0575 31.2375)
3566,36068,31.2875,169,83205,31.289387,121.157952,121.1625,POINT (121.1625 31.2875)
3567,36502,31.1775,74,83205,31.179380,121.067909,121.0725,POINT (121.0725 31.1775)
3568,36758,31.4825,38,83205,31.484439,121.253051,121.2575,POINT (121.2575 31.4825)


In [ ]:
grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp"
population_data = r'D:\shanghai workshop\building+park+waterbody\【25028】基于手机信令的上海市人口分布数据\[20250331]基于手机信令的上海市人口分布数据（2023年）\home.shp'

In [ ]:
import geopandas as gpd
import os

def calculate_average_by_grid(grid_path, point_path, value_field, output_path):
    # 读取数据
    grid = gpd.read_file(grid_path)
    points = gpd.read_file(point_path)

    # 投影统一（必须是米制）
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if not points.crs.is_projected:
        points = points.to_crs(grid.crs)
    elif grid.crs != points.crs:
        points = points.to_crs(grid.crs)

    # 去除无效点
    points = points[points.geometry.notnull() & ~points.geometry.is_empty]
    grid = grid[grid.geometry.notnull() & ~grid.geometry.is_empty]

    # 使用空间连接，将每个点对应到所属网格
    joined = gpd.sjoin(points, grid, how="inner", predicate="within")

    # 统计每个 grid 中的平均值（value_field）
    grouped = joined.groupby("index_right")[value_field].mean()

    # 将统计结果赋值回 grid
    grid["pop_mean"] = grid.index.map(grouped).fillna(0)

    # 保存输出
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    grid.to_file(output_path)

    print(f"✅ 平均值已写入字段 'pop_mean'，输出: {output_path}")

# 调用示例
calculate_average_by_grid(
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    point_path=r"D:\shanghai workshop\building+park+waterbody\【25028】基于手机信令的上海市人口分布数据\[20250331]基于手机信令的上海市人口分布数据（2023年）\home.shp",
    value_field="pop",  # 请替换为实际字段名
    output_path=r"D:\shanghai workshop\vector\shanghai_grid_with_pop_avg.shp"
)
